# Spine 1 · Loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dnakhoa/llm-engineering-playground/blob/main/spine/01-loop/loop.ipynb)

You will run the Flagship Agent's loop on one Acme Notes Case and grade it with
a Check. Everything here runs **Offline**: the model's replies are replayed from
a reviewed recording, so you need no API key and spend nothing.

The lesson that goes with this notebook is `spine/01-loop/README.md`.

In [ ]:
# Setup. Locally this finds the course folder. On Colab it clones the course
# into the current folder once, and finds that clone again when you re-run this
# cell or restart the runtime.
import subprocess
import sys
from pathlib import Path

COURSE_URL = "https://github.com/dnakhoa/llm-engineering-playground.git"
CLONE = Path.cwd() / "llm-engineering-playground"


def find_course(start):
    for folder in [start, *start.parents, CLONE]:
        if (folder / "company").is_dir() and (folder / "llm").is_dir():
            return folder
    return None


COURSE = find_course(Path.cwd())
if COURSE is None:
    subprocess.run(["git", "clone", "--depth", "1", COURSE_URL, str(CLONE)], check=True)
    COURSE = CLONE
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("Course at", COURSE)

## 1. The Case and the Backend

A **Case** is one customer request. This one: a customer on the Free plan asks
to move to Pro. The **Backend** is Acme Notes' deterministic mock of accounts
and plans, reset to the same seed for every Case.

In [ ]:
from company.backend import Backend
from company.runner import load_case

case = load_case("upgrade-to-pro")
print("Customer:", case.customer_account_id)
print("Says:    ", case.opening_message)
print("Expected:", case.expected_state_change)
print()
for account_id, account in Backend.seeded().export_state()["accounts"].items():
    print(account_id, account)

## 2. The loop

This is the whole agent. Ask the model; if it asked for tools, run them and
ask again; stop when it answers without a tool call, or when it runs out of
steps. `env` is what the Case runner hands every agent: the Actions as tools,
and the model.

In [ ]:
import inspect

import flagship.loop

print(inspect.getsource(flagship.loop.run))

## 3. Run the Case, Offline

`run_case` builds a fresh Backend, runs the agent against it, and returns an
**Outcome**: the final state and its diff from the seed, the Actions attempted
and executed, the transcript, usage and cost, and whether the Case was
resolved.

In [ ]:
from company.runner import run_case

outcome = run_case(case, "flagship.loop:run", mode="offline")

for message in outcome.transcript:
    for call in message.tool_calls:
        print("[{}] calls {}({})".format(message.role, call.name, dict(call.arguments)))
    for result in message.tool_results:
        print("[tool] {}".format(result.content))
    if message.text:
        print("[{}] {}".format(message.role, message.text))
print()
print("Diff:     ", outcome.diff)
print("Executed: ", [a.name for a in outcome.actions_executed])
print("Usage:    ", outcome.usage)
print("Cost:     ${:.6f}".format(outcome.cost_usd))
print("Resolved: ", outcome.resolved)

## 4. The Check

A Check asserts on Backend state and on the Actions that actually ran, not on
what the agent said.

In [ ]:
from checks.spine_1_loop import plan_changed_to_pro_exactly_once

check = plan_changed_to_pro_exactly_once(outcome)
print("PASS" if check.passed else "FAIL", "-", check.name, "-", check.detail)

## 5. Authorization lives in the Action

Suppose the customer talks the agent into changing *someone else's* plan. The
prompt says not to, but a prompt can be argued with. The Action cannot: it
knows whose Case this is and refuses anything else.

In [ ]:
from llm.types import ToolCall

actions = Backend.seeded().actions_for("acct_1001")
refused = actions.run(ToolCall(id="c1", name="change_plan",
                               arguments={"account_id": "acct_1002", "plan": "free"}))
print("is_error:", refused.is_error)
print(refused.content)

## 6. The step limit

An agent with no step limit of its own, and a model that keeps asking for the
same tool. The Case runner stops it after `step_limit` model calls and says so
on the Outcome. The stub transport stands in for the model here.

In [ ]:
from llm.testing import StubTransport
from llm.types import Message

LOOK_UP = {
    "content": [{"type": "tool_use", "id": "t1", "name": "look_up_account",
                 "input": {"account_id": "acct_1001"}}],
    "stop_reason": "tool_use",
    "usage": {"input_tokens": 100, "output_tokens": 20},
}


def loops_for_ever(customer_turn, env):
    messages = [Message.user(customer_turn)]
    while True:
        response = env.complete(messages, tools=env.tools)
        messages.append(Message.assistant(response.text or None, response.tool_calls))


looping = run_case(case, loops_for_ever,
                   transport=StubTransport([LOOK_UP] * 20), step_limit=3)
print("Steps:", looping.steps, "of", looping.step_limit)
print("Stopped by the step limit:", looping.step_limit_reached)
print(plan_changed_to_pro_exactly_once(looping).detail)

## 7. Your turn

Write your own `run(customer_turn, env)` and grade it with the same Check.

* An Offline replay answers only the requests it recorded, so your agent will
  get a `ReplayMismatchError` there unless it sends exactly what the reference
  sends. That is the replay doing its job.
* To grade your own agent, run it live on your own key:
  `run_case(case, my_run, mode="live")` with `ANTHROPIC_API_KEY` set, or pass
  `model=` any model in `llm/models.json` with that provider's key.

The cell below runs an agent that chats but never acts, so you can see what an
unresolved Outcome looks like.

In [ ]:
def my_run(customer_turn, env):
    # Replace this with your loop.
    return "Thanks, I'll pass that on!"


mine = run_case(case, my_run, transport=StubTransport())
print("Resolved:", mine.resolved)
print(plan_changed_to_pro_exactly_once(mine).detail)